# Final Pipeline (End-to-End)

This notebook pulls everything together into one automatic workflow. It imports our custom cleaning and modeling scripts from the `src/` folder and runs the whole process from raw data to a finished model evaluation in one pass.


## 1. Imports


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append('..')

# src/ reusable modules
from src.data_cleaning import (
    filter_target_crops,
    remove_invalid_rows,
    handle_missing_values,
    cap_rainfall_outliers
)
from src.feature_engineering import create_yield_per_hectare
from src.scraping import fetch_wikipedia_tables
from src.merge_data import merge_supplementary_data
from src.modeling import (
    prepare_features,
    split_data,
    train_model,
    tune_random_forest,
    evaluate_model,
    cross_validate_model
)

print('All src/ modules imported successfully.')


All src/ modules imported successfully.


## 2. Data Preparation


In [2]:
# Load raw dataset
df_raw = pd.read_csv('../data/raw/crop_yield.csv')
print(f'Raw dataset loaded: {df_raw.shape}')

# Apply cleaning pipeline using src/ functions
df = filter_target_crops(df_raw)
df = remove_invalid_rows(df)
df = handle_missing_values(df)
df = cap_rainfall_outliers(df)
df = create_yield_per_hectare(df)

print(f'After cleaning: {df.shape}')
print('Crops present:', df['Crop'].value_counts().to_dict())


Raw dataset loaded: (19689, 10)
Rainfall outliers capped at 3608.7 mm.
Feature engineering complete: yield_per_hectare added.
After cleaning: (1742, 10)
Crops present: {'Rice': 1197, 'Wheat': 545}


In [3]:
# Scrape and merge supplementary data
df_wiki = fetch_wikipedia_tables()
df = merge_supplementary_data(df, df_wiki)

# Encode categoricals and standardise
from sklearn.preprocessing import StandardScaler

for col in ['Season', 'State', 'Crop', 'Zone']:
    if col in df.columns:
        df[col] = df[col].str.strip()

cat_cols = [c for c in ['Crop', 'Season', 'State', 'Zone'] if c in df.columns]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

num_cols = ['Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide']
valid_num = [c for c in num_cols if c in df.columns]
scaler = StandardScaler()
df[valid_num] = scaler.fit_transform(df[valid_num])

print(f'Final prepared dataset: {df.shape}')


Scraping successful. Extracted 29 states/regions from Wikipedia.
Merging primary dataset with scraped Wikipedia data on [State]...
Merge complete. New dataset shape: (1742, 11)
Final prepared dataset: (1742, 47)


## 3. Modelling


In [4]:
# Prepare features and split
X, y = prepare_features(df, target_col='yield_per_hectare')
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

# Train baseline and tuned models using src/modeling.py
pipeline_lr = train_model(X_train, y_train, model_type='linear_regression')
pipeline_rf = tune_random_forest(X_train, y_train)


Train: 1393 samples | Test: 349 samples
Done training: linear_regression


Best settings found: {'regressor__max_depth': None, 'regressor__n_estimators': 100}
Best R2 score from CV: 0.8900


## 4. Evaluation


In [5]:
# Evaluate both models
lr_metrics = evaluate_model(pipeline_lr, X_test, y_test, 'Linear Regression')
rf_metrics = evaluate_model(pipeline_rf, X_test, y_test, 'Random Forest (Tuned)')

results = pd.DataFrame([lr_metrics, rf_metrics]).set_index('Model')
print('\n=== Final Model Comparison ===')
print(results)

# Cross-validate best model
print('\n=== Cross-Validation ===')
cv_scores = cross_validate_model(pipeline_rf, X, y, cv=5, scoring='r2')


Linear Regression              | MAE=0.3747 | RMSE=0.4990 | R2=0.7148
Random Forest (Tuned)          | MAE=0.1938 | RMSE=0.2977 | R2=0.8985

=== Final Model Comparison ===
                          MAE    RMSE      R2
Model                                        
Linear Regression      0.3747  0.4990  0.7148
Random Forest (Tuned)  0.1938  0.2977  0.8985

=== Cross-Validation ===


CV scores: [0.7584 0.8044 0.887  0.8545 0.672 ]
Average: 0.7952 | Std Dev: 0.0756


## 5. Research Question 2 — Drought Resilience


In [6]:
# Load pre-scraping data for resilience analysis (retains raw Annual_Rainfall values)
df_pre = pd.read_csv('../data/processed/crop_yield_pre_scraping.csv')

threshold = df_pre['Annual_Rainfall'].quantile(0.25)
df_drought = df_pre[df_pre['Annual_Rainfall'] <= threshold]

resilience = df_drought.groupby('Crop')['yield_per_hectare'].agg(
    Mean_Yield='mean',
    Yield_StdDev='std',
    Count='count'
).round(4)

print(f'Deficient Rainfall threshold: {threshold:.1f} mm')
print('\n=== Resilience in Deficient Rainfall Years ===')
print(resilience)
print(f"\nMore resilient crop: {resilience['Yield_StdDev'].idxmin()} (lower yield variance)")


Deficient Rainfall threshold: 996.1 mm

=== Resilience in Deficient Rainfall Years ===
       Mean_Yield  Yield_StdDev  Count
Crop                                  
Rice       2.6507        0.9296    267
Wheat      2.9151        1.4029    170

More resilient crop: Rice (lower yield variance)


## Pipeline Complete

This pipeline ran from raw data to cleaned dataset to trained models to evaluation in a single notebook. Both the business question (R2=0.90 pre-harvest forecasting) and the research question (Rice more resilient than Wheat under drought) are answered. All data transformation and model logic was imported from `src/` reusable modules.
